<a href="https://colab.research.google.com/github/Aditi21-prog/Exoplanet-finder/blob/main/1_0_Exoplanet_Finder_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello space")


In [ ]:
!pip install lightkurve

In [ ]:
import lightkurve, pandas, torch
print("everything works")

In [ ]:
import pandas as pd

url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+kepid,tce_plnt_num,tce_period,tce_time0bk,tce_duration,tce_depth,av_training_set"
       "+from+q1_q17_dr24_tce&format=csv")
tce = pd.read_csv(url)
print(len(tce))
print(tce.head())

In [ ]:
tce = tce[tce.av_training_set.isin(["PC", "AFP", "NTP"])].copy()
tce["planet"] = (tce.av_training_set == "PC").astype(int)
print(len(tce))
print(tce.planet.mean())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# first cut: 10% of stars become the test pile
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
trainval_idx, test_idx = next(gss.split(tce, groups=tce.kepid))
trainval = tce.iloc[trainval_idx]
test = tce.iloc[test_idx]

# second cut: split the rest into train and validation
gss2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx, val_idx = next(gss2.split(trainval, groups=trainval.kepid))
train = trainval.iloc[train_idx]
val = trainval.iloc[val_idx]

print(len(train), len(val), len(test))
print(set(train.kepid) & set(test.kepid))

In [ ]:
import lightkurve as lk

search = lk.search_lightcurve("KIC 11442793", mission="Kepler", cadence="long")
print(len(search))
print(search)

In [ ]:
lc = search.download_all().stitch().remove_nans()
print(len(lc))
print(lc.time.value[:3])

In [ ]:
lc.plot();

In [ ]:
k90 = tce[tce.kepid == 11442793]
print(k90)

In [ ]:
row = k90.iloc[0]
period = row.tce_period
t0 = row.tce_time0bk
dur = row.tce_duration / 24   # table gives hours, we need days

mask = lc.create_transit_mask(period=period, transit_time=t0, duration=2 * dur)
flat = lc.flatten(window_length=401, mask=mask)
folded = flat.fold(period=period, epoch_time=t0)
folded.bin(time_bin_size=dur / 5).scatter();

In [ ]:
import numpy as np

def binned(phase, flux, lo, hi, n):
    edges = np.linspace(lo, hi, n + 1)
    idx = np.digitize(phase, edges) - 1
    out = np.array([np.median(flux[idx == i]) if (idx == i).any() else np.nan
                    for i in range(n)])
    out[np.isnan(out)] = np.nanmedian(out)
    return out

def norm(v):
    v = v - np.median(v)
    d = abs(v.min())
    return v / d if d > 0 else v

def make_views(lc, period, t0, duration_hr):
    dur = duration_hr / 24
    mask = lc.create_transit_mask(period=period, transit_time=t0, duration=2 * dur)
    flat = lc.flatten(window_length=401, mask=mask)
    t, f = flat.time.value, flat.flux.value
    phase = (t - t0 + period / 2) % period - period / 2
    glob = binned(phase, f, -period / 2, period / 2, 2001)
    loc = binned(phase, f, -4 * dur, 4 * dur, 201)
    depth = 1 - np.median(f[np.abs(phase) < dur / 2])
    return norm(glob), norm(loc), depth

print("functions ready")

In [ ]:
import matplotlib.pyplot as plt

row = k90.iloc[0]
g, l, depth = make_views(lc, row.tce_period, row.tce_time0bk, row.tce_duration)

print("global view length:", len(g))
print("local view length:", len(l))
print("depth:", depth)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(g, ".", markersize=3)
axes[0].set_title("Global view (whole orbit)")
axes[1].plot(l, ".", markersize=3)
axes[1].set_title("Local view (zoomed on the dip)")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/exoplanet/lc', exist_ok=True)
print("folder ready")

In [ ]:
import numpy as np, lightkurve as lk, os

LC_DIR = '/content/drive/MyDrive/exoplanet/lc'

def get_lc(kepid):
    path = f"{LC_DIR}/{kepid}.npz"
    if os.path.exists(path):
        d = np.load(path)
        return lk.LightCurve(time=d["t"], flux=d["f"])
    try:
        res = lk.search_lightcurve(f"KIC {kepid}", mission="Kepler", cadence="long")
        if len(res) == 0:
            return None
        lc = res.download_all().stitch().remove_nans()
    except Exception:
        return None
    np.savez(path, t=lc.time.value, f=lc.flux.value)
    return lc

def build(df, name):
    G, L, D, Y = [], [], [], []
    for i, (_, r) in enumerate(df.iterrows()):
        lc = get_lc(int(r.kepid))
        if lc is None:
            continue
        try:
            g, l, d = make_views(lc, r.tce_period, r.tce_time0bk, r.tce_duration)
        except Exception:
            continue
        G.append(g); L.append(l); D.append(d); Y.append(r.planet)
        if i % 20 == 0:
            print(name, i, "of", len(df))
    np.savez(f"/content/drive/MyDrive/exoplanet/{name}.npz",
             G=np.array(G), L=np.array(L), D=np.array(D), Y=np.array(Y))
    print(name, "done:", len(Y), "examples")

print("recipe ready")

In [ ]:
small_train = train[train.kepid.isin(train.kepid.unique()[:140])]
small_val = val[val.kepid.isin(val.kepid.unique()[:30])]
small_test = test[test.kepid.isin(test.kepid.unique()[:30])]
print(len(small_train), len(small_val), len(small_test))

In [ ]:
build(small_val, "val")
build(small_test, "test")
build(small_train, "train")

In [ ]:
import numpy as np

tr = np.load('/content/drive/MyDrive/exoplanet/train.npz')
va = np.load('/content/drive/MyDrive/exoplanet/val.npz')
te = np.load('/content/drive/MyDrive/exoplanet/test.npz')

print("train planets:", tr["Y"].sum(), "of", len(tr["Y"]))
print("val planets:", va["Y"].sum(), "of", len(va["Y"]))
print("test planets:", te["Y"].sum(), "of", len(te["Y"]))
print("shapes:", tr["G"].shape, tr["L"].shape)

In [ ]:
def get_features(df):
    rows = []
    for _, r in df.iterrows():
        lc = get_lc(int(r.kepid))
        if lc is None:
            continue
        try:
            make_views(lc, r.tce_period, r.tce_time0bk, r.tce_duration)
        except Exception:
            continue
        rows.append((r.tce_period, r.tce_duration))
    return np.array(rows)

Ftr = get_features(small_train)
Fva = get_features(small_val)
Fte = get_features(small_test)
print(Ftr.shape, Fva.shape, Fte.shape)

In [ ]:
Xtr = np.column_stack([Ftr, tr["D"]])
Xte = np.column_stack([Fte, te["D"]])

base = HistGradientBoostingClassifier(random_state=42).fit(Xtr, tr["Y"])
probs = base.predict_proba(Xte)[:, 1]
print("baseline test AUC:", roc_auc_score(te["Y"], probs))

In [ ]:
print("train planets:", tr["Y"].sum(), "of", len(tr["Y"]))
print("val planets:  ", va["Y"].sum(), "of", len(va["Y"]))
print("test planets: ", te["Y"].sum(), "of", len(te["Y"]))
print("shapes:", tr["G"].shape, tr["L"].shape)

In [ ]:
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score

torch.manual_seed(42)

def branch(n_blocks):
    layers, ch = [], 1
    for i in range(n_blocks):
        out = 16 * 2**i
        layers += [nn.Conv1d(ch, out, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2)]
        ch = out
    return nn.Sequential(*layers, nn.Flatten())

class PlanetNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.glob = branch(4)   # 2001 -> 125 long, 128 channels
        self.loc = branch(3)    # 201 -> 25 long, 64 channels
        self.head = nn.Sequential(
            nn.Linear(128 * 125 + 64 * 25, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, g, l):
        z = torch.cat([self.glob(g), self.loc(l)], dim=1)
        return self.head(z).squeeze(1)

def to_t(d):
    return (torch.tensor(d["G"], dtype=torch.float32).unsqueeze(1),
            torch.tensor(d["L"], dtype=torch.float32).unsqueeze(1),
            torch.tensor(d["Y"], dtype=torch.float32))

Gtr, Ltr, Ytr = to_t(tr)
Gva, Lva, Yva = to_t(va)
Gte, Lte, Yte = to_t(te)
print("network ready", Gtr.shape, Ltr.shape)

In [ ]:
import copy
model = PlanetNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

best_auc, best_state, wait = 0, None, 0
for epoch in range(40):
    model.train()
    perm = torch.randperm(len(Ytr))
    for i in range(0, len(perm), 32):
        b = perm[i:i+32]
        opt.zero_grad()
        loss = loss_fn(model(Gtr[b], Ltr[b]), Ytr[b])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        vp = torch.sigmoid(model(Gva, Lva)).numpy()
    auc = roc_auc_score(Yva.numpy(), vp)
    print(f"epoch {epoch+1}  loss {loss.item():.3f}  val AUC {auc:.3f}")
    if auc > best_auc:
        best_auc, best_state, wait = auc, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= 8:
            break
model.load_state_dict(best_state)
print("best val AUC:", best_auc)

In [ ]:
model.eval()
with torch.no_grad():
    tp = torch.sigmoid(model(Gte, Lte)).numpy()

cnn_auc = roc_auc_score(Yte.numpy(), tp)
print("CNN test AUC:     ", cnn_auc)
print("Baseline test AUC:", roc_auc_score(te["Y"], probs))

In [ ]:
import numpy as np, os, time

rng = np.random.RandomState(42)
stars = rng.choice(tce.kepid.unique(), size=3000, replace=False)

big_train = train[train.kepid.isin(stars)]
big_val = val[val.kepid.isin(stars)]
big_test = test[test.kepid.isin(stars)]
print(len(stars), len(big_train), len(big_val), len(big_test))

In [ ]:
LIMIT_HOURS = 3
start = time.time()
todo = [int(k) for k in stars if not os.path.exists(f"{LC_DIR}/{int(k)}.npz")]
print(len(todo), "stars still to download")

for n, k in enumerate(todo):
    if time.time() - start > LIMIT_HOURS * 3600:
        print("time limit reached. Run this cell again to continue.")
        break
    get_lc(k)
    if n % 25 == 0:
        print(n, "of", len(todo), f"({(time.time() - start) / 60:.0f} min)")
print("finished this run")

In [ ]:
stars = stars[:1000]

big_train = train[train.kepid.isin(stars)]
big_val = val[val.kepid.isin(stars)]
big_test = test[test.kepid.isin(stars)]
print(len(stars), len(big_train), len(big_val), len(big_test))

In [ ]:
import numpy as np, pandas as pd, os, time
import lightkurve as lk
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive
drive.mount('/content/drive')

# label table
url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+kepid,tce_plnt_num,tce_period,tce_time0bk,tce_duration,tce_depth,av_training_set"
       "+from+q1_q17_dr24_tce&format=csv")
tce = pd.read_csv(url)
tce = tce[tce.av_training_set.isin(["PC", "AFP", "NTP"])].copy()
tce["planet"] = (tce.av_training_set == "PC").astype(int)

# same split as before (same seeds, so same piles)
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
trainval_idx, test_idx = next(gss.split(tce, groups=tce.kepid))
trainval = tce.iloc[trainval_idx]
test = tce.iloc[test_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx, val_idx = next(gss2.split(trainval, groups=trainval.kepid))
train = trainval.iloc[train_idx]
val = trainval.iloc[val_idx]

# same 3000 random stars as before, then keep the first 1000
rng = np.random.RandomState(42)
stars = rng.choice(tce.kepid.unique(), size=3000, replace=False)[:1000]
big_train = train[train.kepid.isin(stars)]
big_val = val[val.kepid.isin(stars)]
big_test = test[test.kepid.isin(stars)]
print(len(stars), len(big_train), len(big_val), len(big_test))

In [ ]:
!pip install lightkurve

In [ ]:
import numpy as np, pandas as pd, os, time
import lightkurve as lk
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive
drive.mount('/content/drive')

# label table
url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+kepid,tce_plnt_num,tce_period,tce_time0bk,tce_duration,tce_depth,av_training_set"
       "+from+q1_q17_dr24_tce&format=csv")
tce = pd.read_csv(url)
tce = tce[tce.av_training_set.isin(["PC", "AFP", "NTP"])].copy()
tce["planet"] = (tce.av_training_set == "PC").astype(int)

# same split as before (same seeds, so same piles)
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
trainval_idx, test_idx = next(gss.split(tce, groups=tce.kepid))
trainval = tce.iloc[trainval_idx]
test = tce.iloc[test_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx, val_idx = next(gss2.split(trainval, groups=trainval.kepid))
train = trainval.iloc[train_idx]
val = trainval.iloc[val_idx]

# same 3000 random stars as before, then keep the first 1000
rng = np.random.RandomState(42)
stars = rng.choice(tce.kepid.unique(), size=3000, replace=False)[:1000]
big_train = train[train.kepid.isin(stars)]
big_val = val[val.kepid.isin(stars)]
big_test = test[test.kepid.isin(stars)]
print(len(stars), len(big_train), len(big_val), len(big_test))

In [ ]:
import numpy as np, lightkurve as lk, os

LC_DIR = '/content/drive/MyDrive/exoplanet/lc'

def binned(phase, flux, lo, hi, n):
    edges = np.linspace(lo, hi, n + 1)
    idx = np.digitize(phase, edges) - 1
    out = np.array([np.median(flux[idx == i]) if (idx == i).any() else np.nan
                    for i in range(n)])
    out[np.isnan(out)] = np.nanmedian(out)
    return out

def norm(v):
    v = v - np.median(v)
    d = abs(v.min())
    return v / d if d > 0 else v

def make_views(lc, period, t0, duration_hr):
    dur = duration_hr / 24
    mask = lc.create_transit_mask(period=period, transit_time=t0, duration=2 * dur)
    flat = lc.flatten(window_length=401, mask=mask)
    t, f = flat.time.value, flat.flux.value
    phase = (t - t0 + period / 2) % period - period / 2
    glob = binned(phase, f, -period / 2, period / 2, 2001)
    loc = binned(phase, f, -4 * dur, 4 * dur, 201)
    depth = 1 - np.median(f[np.abs(phase) < dur / 2])
    return norm(glob), norm(loc), depth

def get_lc(kepid):
    path = f"{LC_DIR}/{kepid}.npz"
    if os.path.exists(path):
        d = np.load(path)
        return lk.LightCurve(time=d["t"], flux=d["f"])
    try:
        res = lk.search_lightcurve(f"KIC {kepid}", mission="Kepler", cadence="long")
        if len(res) == 0:
            return None
        lc = res.download_all().stitch().remove_nans()
    except Exception:
        return None
    np.savez(path, t=lc.time.value, f=lc.flux.value)
    return lc

print("recipes ready")

In [ ]:
!pip install -q lightkurve

import numpy as np, pandas as pd, os, time
import lightkurve as lk
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive
drive.mount('/content/drive')

LC_DIR = '/content/drive/MyDrive/exoplanet/lc'
os.makedirs(LC_DIR, exist_ok=True)

# label table
url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+kepid,tce_plnt_num,tce_period,tce_time0bk,tce_duration,tce_depth,av_training_set"
       "+from+q1_q17_dr24_tce&format=csv")
tce = pd.read_csv(url)
tce = tce[tce.av_training_set.isin(["PC", "AFP", "NTP"])].copy()
tce["planet"] = (tce.av_training_set == "PC").astype(int)

# same split as before
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
trainval_idx, test_idx = next(gss.split(tce, groups=tce.kepid))
trainval = tce.iloc[trainval_idx]
test = tce.iloc[test_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx, val_idx = next(gss2.split(trainval, groups=trainval.kepid))
train = trainval.iloc[train_idx]
val = trainval.iloc[val_idx]

# same random stars as before, first 1000
rng = np.random.RandomState(42)
stars = rng.choice(tce.kepid.unique(), size=3000, replace=False)[:1000]
big_train = train[train.kepid.isin(stars)]
big_val = val[val.kepid.isin(stars)]
big_test = test[test.kepid.isin(stars)]
print(len(stars), len(big_train), len(big_val), len(big_test))

# recipes
def binned(phase, flux, lo, hi, n):
    edges = np.linspace(lo, hi, n + 1)
    idx = np.digitize(phase, edges) - 1
    out = np.array([np.median(flux[idx == i]) if (idx == i).any() else np.nan
                    for i in range(n)])
    out[np.isnan(out)] = np.nanmedian(out)
    return out

def norm(v):
    v = v - np.median(v)
    d = abs(v.min())
    return v / d if d > 0 else v

def make_views(lc, period, t0, duration_hr):
    dur = duration_hr / 24
    mask = lc.create_transit_mask(period=period, transit_time=t0, duration=2 * dur)
    flat = lc.flatten(window_length=401, mask=mask)
    t, f = flat.time.value, flat.flux.value
    phase = (t - t0 + period / 2) % period - period / 2
    glob = binned(phase, f, -period / 2, period / 2, 2001)
    loc = binned(phase, f, -4 * dur, 4 * dur, 201)
    depth = 1 - np.median(f[np.abs(phase) < dur / 2])
    return norm(glob), norm(loc), depth

def get_lc(kepid):
    path = f"{LC_DIR}/{kepid}.npz"
    if os.path.exists(path):
        d = np.load(path)
        return lk.LightCurve(time=d["t"], flux=d["f"])
    try:
        res = lk.search_lightcurve(f"KIC {kepid}", mission="Kepler", cadence="long")
        if len(res) == 0:
            return None
        lc = res.download_all().stitch().remove_nans()
    except Exception:
        return None
    np.savez(path, t=lc.time.value, f=lc.flux.value)
    return lc

print("everything ready")

In [ ]:
LIMIT_HOURS = 3
start = time.time()
todo = [int(k) for k in stars if not os.path.exists(f"{LC_DIR}/{int(k)}.npz")]
print(len(todo), "stars still to download")

for n, k in enumerate(todo):
    if time.time() - start > LIMIT_HOURS * 3600:
        print("time limit reached. Run this cell again to continue.")
        break
    get_lc(k)
    if n % 25 == 0:
        print(n, "of", len(todo), f"({(time.time() - start) / 60:.0f} min)")
print("finished this run")

In [ ]:
def build2(df, name):
    G, L, D, Y, P, U = [], [], [], [], [], []
    for i, (_, r) in enumerate(df.iterrows()):
        lc = get_lc(int(r.kepid))
        if lc is None:
            continue
        try:
            g, l, d = make_views(lc, r.tce_period, r.tce_time0bk, r.tce_duration)
        except Exception:
            continue
        G.append(g); L.append(l); D.append(d); Y.append(r.planet)
        P.append(r.tce_period); U.append(r.tce_duration)
        if i % 100 == 0:
            print(name, i, "of", len(df))
    np.savez(f"/content/drive/MyDrive/exoplanet/{name}_big.npz",
             G=np.array(G), L=np.array(L), D=np.array(D), Y=np.array(Y),
             P=np.array(P), U=np.array(U))
    print(name, "done:", len(Y), "examples,", int(np.sum(Y)), "planets")

build2(big_val, "val")
build2(big_test, "test")
build2(big_train, "train")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

tr = np.load('/content/drive/MyDrive/exoplanet/train_big.npz')
va = np.load('/content/drive/MyDrive/exoplanet/val_big.npz')
te = np.load('/content/drive/MyDrive/exoplanet/test_big.npz')

def feats(d):
    return np.column_stack([d["P"], d["U"], d["D"]])

base = HistGradientBoostingClassifier(random_state=42).fit(feats(tr), tr["Y"])
base_probs = base.predict_proba(feats(te))[:, 1]
base_auc = roc_auc_score(te["Y"], base_probs)
base_pred = (base_probs > 0.5).astype(int)

print("baseline test AUC:      ", round(base_auc, 4))
print("baseline test precision:", round(precision_score(te["Y"], base_pred), 4))
print("baseline test recall:   ", round(recall_score(te["Y"], base_pred), 4))

In [ ]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
print("ready")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

tr = np.load('/content/drive/MyDrive/exoplanet/train_big.npz')
va = np.load('/content/drive/MyDrive/exoplanet/val_big.npz')
te = np.load('/content/drive/MyDrive/exoplanet/test_big.npz')

def feats(d):
    return np.column_stack([d["P"], d["U"], d["D"]])

base = HistGradientBoostingClassifier(random_state=42).fit(feats(tr), tr["Y"])
base_probs = base.predict_proba(feats(te))[:, 1]
base_auc = roc_auc_score(te["Y"], base_probs)
base_pred = (base_probs > 0.5).astype(int)

print("baseline test AUC:      ", round(base_auc, 4))
print("baseline test precision:", round(precision_score(te["Y"], base_pred), 4))
print("baseline test recall:   ", round(recall_score(te["Y"], base_pred), 4))

In [ ]:
import torch, torch.nn as nn, copy
torch.manual_seed(42)

def branch(n_blocks):
    layers, ch = [], 1
    for i in range(n_blocks):
        out = 16 * 2**i
        layers += [nn.Conv1d(ch, out, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2)]
        ch = out
    return nn.Sequential(*layers, nn.Flatten())

class PlanetNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.glob = branch(4)
        self.loc = branch(3)
        self.head = nn.Sequential(
            nn.Linear(128 * 125 + 64 * 25, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, g, l):
        z = torch.cat([self.glob(g), self.loc(l)], dim=1)
        return self.head(z).squeeze(1)

def to_t(d):
    return (torch.tensor(d["G"], dtype=torch.float32).unsqueeze(1),
            torch.tensor(d["L"], dtype=torch.float32).unsqueeze(1),
            torch.tensor(d["Y"], dtype=torch.float32))

Gtr, Ltr, Ytr = to_t(tr)
Gva, Lva, Yva = to_t(va)
Gte, Lte, Yte = to_t(te)

model = PlanetNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

best_auc, best_state, wait = 0, None, 0
for epoch in range(60):
    model.train()
    perm = torch.randperm(len(Ytr))
    for i in range(0, len(perm), 32):
        b = perm[i:i+32]
        opt.zero_grad()
        loss = loss_fn(model(Gtr[b], Ltr[b]), Ytr[b])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        vp = torch.sigmoid(model(Gva, Lva)).numpy()
    auc = roc_auc_score(Yva.numpy(), vp)
    print(f"epoch {epoch+1}  loss {loss.item():.3f}  val AUC {auc:.3f}")
    if auc > best_auc:
        best_auc, best_state, wait = auc, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= 10:
            break
model.load_state_dict(best_state)
print("best val AUC:", best_auc)

In [ ]:
model.eval()
with torch.no_grad():
    cnn_probs = torch.sigmoid(model(Gte, Lte)).numpy()
cnn_pred = (cnn_probs > 0.5).astype(int)

print("CNN test AUC:      ", round(roc_auc_score(te["Y"], cnn_probs), 4))
print("CNN test precision:", round(precision_score(te["Y"], cnn_pred), 4))
print("CNN test recall:   ", round(recall_score(te["Y"], cnn_pred), 4))
print()
print("Baseline test AUC: ", round(base_auc, 4))
print("Baseline precision:", round(precision_score(te["Y"], base_pred), 4))
print("Baseline recall:   ", round(recall_score(te["Y"], base_pred), 4))

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/exoplanet/cnn_1000stars.pt')
print("model saved")

In [ ]:
order = np.argsort(cnn_probs)
print("lowest-probability non-planets (indices):",
      [int(i) for i in order if te["Y"][i] == 0][:5])
print("highest-probability planets (indices):",
      [int(i) for i in order[::-1] if te["Y"][i] == 1][:5])

In [ ]:
import matplotlib.pyplot as plt

def label_of(i):
    m = big_test[np.isclose(big_test.tce_period, te["P"][i]) &
                 np.isclose(big_test.tce_duration, te["U"][i])]
    return m.iloc[0].av_training_set, int(m.iloc[0].kepid)

order = np.argsort(cnn_probs)
nonplanets = [int(i) for i in order if te["Y"][i] == 0]
planets = [int(i) for i in order[::-1] if te["Y"][i] == 1]

pl = planets[0]
eb = next((i for i in nonplanets[:40] if label_of(i)[0] == "AFP"), nonplanets[0])

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for row, (idx, name) in enumerate([(pl, "Planet candidate"), (eb, "False positive")]):
    lab, kid = label_of(idx)
    p = cnn_probs[idx]
    axes[row, 0].plot(te["G"][idx], ".", markersize=2)
    axes[row, 0].set_title(f"{name} (KIC {kid}, label {lab}): global view")
    axes[row, 1].plot(te["L"][idx], ".", markersize=3)
    axes[row, 1].set_title(f"local view, CNN planet probability = {p:.2f}")
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/exoplanet/readme_visual.png', dpi=150)
plt.show()
print("planet index:", pl, "| false positive index:", eb, "| its label:", label_of(eb)[0])

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+kepid,tce_plnt_num,tce_period,tce_time0bk,tce_duration,tce_depth,av_training_set"
       "+from+q1_q17_dr24_tce&format=csv")
tce = pd.read_csv(url)
tce = tce[tce.av_training_set.isin(["PC", "AFP", "NTP"])].copy()
tce["planet"] = (tce.av_training_set == "PC").astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
trainval_idx, test_idx = next(gss.split(tce, groups=tce.kepid))
test = tce.iloc[test_idx]

rng = np.random.RandomState(42)
stars = rng.choice(tce.kepid.unique(), size=3000, replace=False)[:1000]
big_test = test[test.kepid.isin(stars)]
print(len(big_test))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for k, idx in enumerate(planets[:3]):
    lab, kid = label_of(idx)
    axes[0, k].plot(te["L"][idx], ".", markersize=3)
    axes[0, k].set_title(f"idx {idx}, KIC {kid}, {lab}, p={cnn_probs[idx]:.2f}")
afp = [i for i in nonplanets if label_of(i)[0] == "AFP"][:3]
for k, idx in enumerate(afp):
    lab, kid = label_of(idx)
    axes[1, k].plot(te["L"][idx], ".", markersize=3)
    axes[1, k].set_title(f"idx {idx}, KIC {kid}, {lab}, p={cnn_probs[idx]:.2f}")
plt.tight_layout()
plt.show()

In [ ]:
pl, fp = 134, 126

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for row, (idx, name) in enumerate([(pl, "Planet candidate"), (fp, "False positive")]):
    lab, kid = label_of(idx)
    axes[row, 0].plot(te["G"][idx], ".", markersize=2)
    axes[row, 0].set_title(f"{name} (KIC {kid}, {lab}): global view")
    axes[row, 1].plot(te["L"][idx], ".", markersize=3)
    axes[row, 1].set_title(f"local view, CNN planet probability = {cnn_probs[idx]:.2f}")
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/exoplanet/readme_visual.png', dpi=150)
plt.show()

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/exoplanet/cnn_1000stars.pt')
print("model saved")